<br/>

<div align="center">
<span style="font-size: 2.5em;">XENON Preprocessing Step 1: PyTorch to CSV</span>
<br/>
<span style="font-size: 1.2em; color: gray;">Convert .pt dataset files to CSV format for XENON fuse pipeline compatibility</span>
</div>

## Background

The XENON fuse simulation container does not include PyTorch, so `.pt` files cannot be loaded directly. This notebook extracts recoil energies from WimPyDD-generated spectra and converts them to CSV format for compatibility with the fuse pipeline.

**Strategy**: Only datasets with low $c_p$ values (low event counts) are processed to save computational resources and focus on the physically relevant low-statistics regime.

## Setup and Imports

In [ ]:
# =============================
# IMPORTS AND PATH SETTINGS
# =============================

import os
import csv
import numpy as np
import torch

desired_root_name = "xenon-sbi"  
while os.path.basename(os.getcwd()) != desired_root_name:
    os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

## Conversion Function

In [2]:
def convert_pt_to_csv(input_path: str, output_path: str) -> None:
    """Convert .pt file containing event_list into .csv file.
    
    Parameters
    ----------
    input_path : str
        Path to input .pt file (must contain 'events' key).
    output_path : str
        Path to output .csv file (directory will be created if needed).
    """
    data = torch.load(input_path, map_location="cpu", weights_only=False)
    events_list = data.get("events", None)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, "w", newline="") as f:
        writer = csv.writer(f)
        for events in events_list:
            if isinstance(events, np.ndarray):
                writer.writerow(events.tolist() if len(events) > 0 else [])
            else:
                writer.writerow([])

    print(f"Saved: {output_path}")

## Configuration: Files to Convert

In [6]:
# Specify files to convert (low-cp datasets only)
files_to_convert = [
    "data/datasets/wimpy/default/wimpy_n300000_low_default.pt",
    "data/datasets/wimpy/x1/wimpy_n300000_low_x1.pt",
    "data/datasets/wimpy/x2/wimpy_n300000_low_x2.pt",
]

# Output directory for CSV files
OUTPUT_BASE = "data/datasets/xenon/wimpy"

## Run Conversion

In [7]:
for pt_file in files_to_convert:
    filename = os.path.basename(pt_file).replace(".pt", ".csv")
    output_path = os.path.join(OUTPUT_BASE, filename)
    convert_pt_to_csv(pt_file, output_path)

print(f"\nConversion complete! {len(files_to_convert)} files processed.")

Saved: data/datasets/xenon/wimpy\wimpy_n300000_low_default.csv
Saved: data/datasets/xenon/wimpy\wimpy_n300000_low_x1.csv
Saved: data/datasets/xenon/wimpy\wimpy_n300000_low_x2.csv

Conversion complete! 3 files processed.
